# COMP0005 - GROUP COURSEWORK
# Experimental Evaluation of Search Data Structures and Algorithms

The cell below defines **AbstractSearchInterface**, an interface to support basic insert/search operations; you will need to implement this three times, to realise your three search data structures of choice among: (1) *2-3 Tree*, (2) *AVL Tree*, (3) *LLRB BST*; (4) *B-Tree*; and (5) *Scapegoat Tree*. <br><br>**Do NOT modify the next cell** - use the dedicated cells further below for your implementation instead. <br>

In [ ]:
# DO NOT MODIFY THIS CELL

from abc import ABC, abstractmethod  

class AbstractSearchInterface(ABC):
    '''
    Abstract class to support search/insert operations (plus underlying data structure)
    
    '''
        
    @abstractmethod
    def insertElement(self, element):     
        '''
        Insert an element in a search tree
            Parameters:
                    element: string to be inserted in the search tree (string)

            Returns:
                    "True" after successful insertion, "False" if element is already present (bool)
        '''
        
        pass 
    

    @abstractmethod
    def searchElement(self, element):
        '''
        Search for an element in a search tree
            Parameters:
                    element: string to be searched in the search tree (string)

            Returns:
                    "True" if element is found, "False" otherwise (bool)
        '''

        pass

Use the cell below to define any auxiliary data structure and python function you may need. Leave the implementation of the main API to the next code cells instead.

In [ ]:
# ADD AUXILIARY DATA STRUCTURE DEFINITIONS AND HELPER CODE HERE

class AvlNode:
    def __init__(self, key: str):
        self.key: str = key
        self.height: int = 0
        self.left: AvlNode | None = None
        self.right: AvlNode | None = None


class LLRBNode:
    def __init__(self, key):
        self.key: str = key
        self.left: LLRBNode | None = None
        self.right: LLRBNode | None = None
        self.color: bool = True # True: red, False: black

class LLRBHelper:

    @staticmethod
    def rotate_left(root: LLRBNode) -> LLRBNode:
        right = root.right
        root.right = right.left
        right.left = root

        right.color = root.color
        root.color = True

        return right

    @staticmethod
    def rotate_right(root: LLRBNode) -> LLRBNode:
        left = root.left
        root.left = left.right
        left.right = root

        left.color = root.color
        root.color = True

        return left

    @staticmethod
    def flip_colors(node: LLRBNode) -> LLRBNode:
        node.color = not node.color
        node.left.color = not node.left.color
        node.right.color = not node.right.color
        return node

    @staticmethod
    def is_red(node: LLRBNode | None) -> bool:
        return node is not None and node.color

    @staticmethod
    def search(key: str, root: LLRBNode | None) -> bool:
        while root:
            if key < root.key:
                root = root.left
            elif key > root.key:
                root = root.right
            else:
                return True
        return False

    @staticmethod
    def insert(key: str, root: LLRBNode | None) -> LLRBNode:
        if not root:
            return LLRBNode(key)
        if key < root.key:
            root.left = LLRBHelper.insert(key, root.left)
        elif key > root.key:
            root.right = LLRBHelper.insert(key, root.right)
        else:
            return root
        if LLRBHelper.is_red(root.right) and not LLRBHelper.is_red(root.left):
            root = LLRBHelper.rotate_left(root)
        if LLRBHelper.is_red(root.left) and LLRBHelper.is_red(root.left.left):
            root = LLRBHelper.rotate_right(root)
        if LLRBHelper.is_red(root.left) and LLRBHelper.is_red(root.right):
            root = LLRBHelper.flip_colors(root)

        return root

class TwoThreeNode:
  '''A node in a 2-3 tree.

  Attributes:
    keys: A sorted list of one or two keys (can have three keys but only temporarily)
    children: A list of zero (leaf), two (2-node) or three (3-node) child nodes
  '''
  def __init__(self, keys=None, children=None):
    self.keys = keys or []
    self.children = children or []
  def is_leaf(self):
    return len(self.children) == 0

Use the cell below to implement the requested API by means of **2-3 Tree** (if among your chosen data structure).

In [ ]:
'''
2-3 Tree Implementation:

A 2-3 tree is a balanced search tree in which
- 2 nodes have one key and two children
- 3 nodes have two keys and three children
- All the leaf nodes lie at the same depth

The tree is balanced by splitting 'overfull' nodes (temporary nodes with 3 keys and 4 children) into two 2 nodes, and passing the middle key up the tree. The tree depth can only be increased by a split operation at the root node: this is how the tree maintains balance.
Splitting is a constant time operation and only acts locally in the structure.

Searching a 2-3 tree of N nodes is an O(logN) operation; inserting a node into a 2-3 tree of size N is also an O(logN) operation. The depth of the tree ranges from log N / log 2 (log base 2 of N), to log N / log 3 (log base 3 of N).
'''

class TwoThreeTree(AbstractSearchInterface):
        
  '''A 2-3 balanced search tree with 'search' and 'insert' operations.
  '''
  def __init__(self, root=None):
    self.root = root

  def searchElement(self, element):
    '''Returns true if and only if the 2-3 tree contains a specified element. Delegates work to recursive helper, 'search'.

    Args:
        node (Node): the root of the tree/subtree to search
        element (_): the element to search for
    '''
    if self.root is None:
      return False
    return self.__search(self.root, element)
  
  def __search(self, node, element):
    '''Returns true if and only if the subtree rooted at a given node contains a specified element.

    Args:
        node (Node): the root of the tree/subtree to search
        element (_): the element to search for
    
    '''

    if node is None:
      return False
    
    # Checks if the element is in this node
    if element in node.keys:
      return True
    
    # Leaf reached without finding the element -> the element is not in the tree
    if node.is_leaf():
      return False
    
    # Pick the correct child to descend the tree to
    # The index of the first key greater than the element is the index of the correct child
    # (If no key is greater, then descend to the rightmost child)
    child_index = len(node.keys)
    for i, key in enumerate(node.keys):
      if element < key:
        child_index = i
        break

    return self.__search(node.children[child_index], element)
  
  def insertElement(self, element):
    '''Insert a element into a 2-3 tree, keeping the balance in check. Delegates work to the recursive _insert method.

    Args:
        element (_): the element to insert
    Returns:
        True if and only if the element is successfully inserted into the tree; False otherwise (if there are duplicates)
    '''
    # If the tree is empty, then create a single leaf root
    if self.root is None:
      self.root = TwoThreeNode([element])
      return True
    
    result = self.__insertElement(self.root, element)
    
    # Result is false -> element already in the tree -> return false
    if result is False:
      return False
    
    # If the root split, then create a new one, one level higher
    # This is the only way the tree can grow taller
    if result is not None:
      middle_key, left_child, right_child = result
      self.root = TwoThreeNode([middle_key], [left_child, right_child])
    
    return True 
  
  def __insertElement(self, node, element):
    '''Insert a element into a 2-3 subtree rooted at a given node, keeping the balance in check. This is a recurisve helper method for insert.

    Args:
        node (Node): the root node of the subtree
        element (_): the element to insert

    Returns:
        None : if the insertion is absorbed by the node without causing a split operation
        middle_key, left_child, right_child (_, Node, Node) : if the node split, the calling method must absorb the key being passed up, and the left and right children
        False : if the element being inserted is already in the tree
    '''
    # Element already in tree -> return false
    if element in node.keys:
      return False
    
    # Base case - leaf node
    if node.is_leaf():

      node.keys.append(element)
      node.keys.sort()

      if len(node.keys) <= 2: # Still a valid 2- or 3- node
        return None
      
      # Overfull (has 3 keys) -> split and pass the middle key up the call chain
      middle_key = node.keys[1]
      left_child = TwoThreeNode([node.keys[0]])
      right_child = TwoThreeNode([node.keys[2]])
      return (middle_key, left_child, right_child)
          
    # Recursive case - 2- or 3- (non-leaf) node
    # Find the right child to recurse to (same logic as in search)
    child_index = len(node.keys)
    for i, key in enumerate(node.keys):
      if element < key:
        child_index = i
        break

    result = self.__insertElement(node.children[child_index], element)
    
    # result False -> duplicate data -> propagate this up the call stack 
    if result is False:
      return False
    if result is None: # Child absorbed the insert without splitting - so do nothing
      return None   
    
    # A child split, so absorb the recieved key and new children
    middle_key, left_child, right_child = result
    node.keys.append(middle_key)
    node.keys.sort()
    # Replace the child at child_index with the two new children
    node.children = node.children[:child_index] + [left_child, right_child] + node.children[child_index+1:]

    if len(node.keys) <= 2: # Still a valid 2- or 3- node - so do not initiate a split
      return None
    # Overfull (has 3 keys) -> split and pass (as before) but attach children
    # left_child node gets the first two children, right_child node get the last two
    middle_key = node.keys[1] 
    left_child = TwoThreeNode([node.keys[0]], node.children[:2])
    right_child = TwoThreeNode([node.keys[2]], node.children[2:])
    return (middle_key, left_child, right_child)

In [ ]:
'''Extremely basic test (temporary)
'''

from random import shuffle 
bst = TwoThreeTree()


shuffled_even_integers = [i for i in range(0, 100, 2)]
shuffle(shuffled_even_integers)

for i in shuffled_even_integers:
  bst.insertElement(i)

positives = [] # Expect all even integers 0 - 100
negatives = [] # Expect all odd integers 0 - 100
integer_range = [i for i in range(100)]
for i in integer_range:
  if bst.searchElement(i):
    positives += [i]
  else:
    negatives += [i]

print("The elements in the tree (should be even integers 0-100) are: ", positives)
print("The elements not in the tree (should be odd integers 0-100) are: ", negatives)

# Note: I wrote this to verify my implementation works - we need to agree on a rigorous testing framework

Use the cell below to implement the requested API by means of **AVL Tree** (if among your chosen data structure).

In [ ]:
class AVLTree(AbstractSearchInterface):

    @staticmethod
    def heightOf(root: AvlNode | None) -> int:
        # height of root = edges from root to furthest leaf
        # -1 because height(leaf) = 1 + max(-1,-1) = 0
        if root is None:
            return -1
        else:
            return 1 + max(
                root.left.height if root.left is not None else -1,
                root.right.height if root.right is not None else -1
            )


    @staticmethod
    def rotateRight(root: AvlNode) -> AvlNode:
        # returns the new root of the subtree (currently root.left)
        assert root.left is not None, "Right rotation requires root.left to be a Node"
        
        # perform rotation
        left = root.left
        leftRight = root.left.right
        root.left = leftRight
        left.right = root

        #update heights
        root.height = AVLTree.heightOf(root)
        left.height = AVLTree.heightOf(left)

        return left


    @staticmethod
    def rotateLeft(root: AvlNode) -> AvlNode:
        # returns the new root of the subtree (root.right)
        assert root.right is not None, "Left rotation requires root.right to be a Node"

        # perform rotation
        right = root.right
        rightLeft = root.right.left
        root.right = rightLeft
        right.left = root

        # update heights
        root.height = AVLTree.heightOf(root)
        right.height = AVLTree.heightOf(right)

        return right   
    

    @staticmethod
    def searchHelper(element: str, root: AvlNode | None) -> bool:
        # returns whether the element is in root's subtree
        if root == None:
            return False
        elif root.key == element:
            return True
        elif root.key > element:
            return AVLTree.searchHelper(element, root.left)
        else:
            return AVLTree.searchHelper(element, root.right)
    

    @staticmethod
    def insertHelper(element: str, root: AvlNode | None) -> AvlNode:
        # insert into the right/left subtree
        if root is None:
            return AvlNode(element)
        elif root.key > element:
            root.left = AVLTree.insertHelper(element, root.left)
        elif root.key < element:
            root.right = AVLTree.insertHelper(element, root.right)

        # root.left and root.right are now balanced
        # height of child nodes may have changed, so height of root may have changed
        root.height = AVLTree.heightOf(root)        

        # balance of root = left height - right height
        # valid AVL tree has valid balance values of -1,0,1
        balance = AVLTree.heightOf(root.left) - AVLTree.heightOf(root.right)

        # case 1: left subtree too heavy and imbalance from left's left subtree
        if balance > 1 and root.left is not None and element < root.left.key:
            return AVLTree.rotateRight(root)
        # case 2: left subtree too heavy and imbalance from left's right subtree
        elif balance > 1 and root.left is not None and element > root.left.key:
            root.left = AVLTree.rotateLeft(root.left)
            return AVLTree.rotateRight(root)
        # case 3: mirror case 1
        if balance < -1 and root.right is not None and element > root.right.key:
            return AVLTree.rotateLeft(root)
        # case 4: mirror case 2
        elif balance < -1 and root.right is not None and element < root.right.key:
            root.right = AVLTree.rotateRight(root.right)
            return AVLTree.rotateLeft(root)
        # else balanced
        else:
            return root
    

    @staticmethod
    def printBfs(root: AvlNode) -> None:
        q: list[AvlNode | None] = [root]
        while len(q) > 0:
            node = q.pop(0)
            if node:
                print(node.key)
                q.append(node.left)
                q.append(node.right)


    def __init__(self, root: AvlNode):
        self.root: AvlNode = root

    
    def insertElement(self, element: str) -> bool:
        canInsert = not AVLTree.searchHelper(element, self.root)
        if canInsert:
            self.root = AVLTree.insertHelper(element, self.root)
        return canInsert
    
    
    def searchElement(self, element: str) -> bool:             
        found = AVLTree.searchHelper(element, self.root)
        return found

In [ ]:
# tmp test block

r = AvlNode("a")
t = AVLTree(r)
for i in range(98, 98+14):
  t.insertElement(chr(i))
AVLTree.printBfs(t.root)

Use the cell below to implement the requested API by means of **LLRB BST** (if among your chosen data structure).

In [ ]:
class LLRBBST(AbstractSearchInterface):

    def __init__(self, root=None):
        self.root = root
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
        if self.searchElement(element):
            return False
        self.root = LLRBHelper.insert(element, self.root)
        self.root.color = False
        inserted = True
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE
        found = LLRBHelper.search(element, self.root)
        
        return found

In [ ]:
# Test
def print_tree(node, indent="", side="root"):
    if node is None:
        print(f"{indent}[{side}] None")
        return

    color = "R" if node.color else "B"
    print(f"{indent}[{side}] {node.key}({color})")
    print_tree(node.left, indent + "   ", "L")
    print_tree(node.right, indent + "   ", "R")

tree = LLRBBST()

print(tree.insertElement("10"))
print(tree.insertElement("20"))
print(tree.insertElement("30"))
print(tree.insertElement("20"))

print(tree.searchElement("10"))
print(tree.searchElement("20"))
print(tree.searchElement("30"))
print(tree.searchElement("40"))

print("root color is black:", tree.root.color == False)

print_tree(tree.root)

Use the cell below to implement the requested API by means of **B-Tree** (if among your chosen data structure).

In [ ]:
class BTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found

Use the cell below to implement the requested API by means of **Scapegoat Tree** (if among your chosen data structure).

In [ ]:
class ScapegoatTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found 

Use the cell below to implement the **synthetic data generator** needed by your experimental framework (be mindful of code readability and reusability).

In [ ]:
import string
import random

class TestDataGenerator():
    '''
    A class to represent a synthetic data generator.

    ...

    Attributes
    ----------
    
    [to be defined as part of the coursework]

    Methods
    -------
    
    [to be defined as part of the coursework]

    '''
    
    #ADD YOUR CODE HERE
    
    def __init__(self, keyLB: int = 0, keyUB: int = 10**6):
        self.__lowerBound: int = keyLB
        self.__upperBound: int = keyUB
        self.__normal_mean: float = (self.__lowerBound + self.__upperBound) / 2
        self.__normal_sd: float = (self.__upperBound - self.__normal_mean) / 3 


    def random_uniform_value(self) -> str:
        value = random.randint(self.__lowerBound, self.__upperBound)
        return str(value)
    

    def random_normal_value(self) -> str:
        # 99.7% of values are within bounds (due to calculation of s.d)
        # reroll if it falls outside those bounds
        while True:
            value = random.gauss(self.__normal_mean, self.__normal_sd)
            if self.__lowerBound <= value <= self.__upperBound:
                return str(value)


    def next_sorted_value(self, n: int):
        interval = (self.__upperBound - self.__lowerBound) / n
        for i in range(n):
            yield str(self.__lowerBound + i * interval)

    
    def next_reverse_sorted_value(self, n: int):
        interval = (self.__upperBound - self.__lowerBound) / n
        for i in range(n-1, -1, -1):
            yield str(self.__lowerBound + i * interval)

    
    def constant_value(self, n: int):
        return str(n)

Use the cell below to implement the requested **experimental framework** (be mindful of code readability and reusability).

In [ ]:
import timeit
import matplotlib

class ExperimentalFramework():
    '''
    A class to represent an experimental framework.

    ...

    Attributes
    ----------
    
    [to be defined as part of the coursework]

    Methods
    -------
    
    [to be defined as part of the coursework]

    '''
            
    #ADD YOUR CODE HERE
    
    def __init__(self):
        pass

    

Use the cell below to illustrate the python code you used to **fully evaluate** your three chosen search data structures and algortihms. The code below should illustrate, for example, how you made used of the **TestDataGenerator** class to generate test data of various size and properties; how you instatiated the **ExperimentalFramework** class to  evaluate each data structure using such data, collect information about their execution time, plot results, etc. Any results you illustrate in the companion PDF report should have been generated using the code below.

In [ ]:
# ADD YOUR TEST CODE HERE 



